### **Previsão de Churn — Carteira Atual (`inference.csv`)**
#### **Grupo 2 · Hackathon Databricks**

*Continuação do `modelo_churn.ipynb`. Lá o modelo foi treinado, validado e salvo; aqui ele é aplicado nos
clientes da carteira atual (`inference.csv`), que ainda não têm a resposta (`Churn`). O resultado é uma
nova base, `inference_com_previsao.csv`, com a probabilidade de saída e a indicação de risco de cada
cliente, pronta para o time de vendas priorizar as ligações de retenção.*

**Decisões desta etapa:**

| Decisão | Por quê |
|---|---|
| **O modelo não é treinado de novo** | O `modelo_churn.joblib` já guarda o pipeline treinado com todo o histórico, o limiar e as colunas esperadas. Carregar o arquivo garante que a previsão usa exatamente o modelo validado. |
| **A limpeza e a criação de variáveis são repetidas aqui** | O pipeline salvo só faz imputação, padronização e One-Hot Encoding. As seções 4 e 5 do `modelo_churn.ipynb` acontecem antes dele, então a carteira precisa passar pelos mesmos passos. |
| **Limiar 0,20** | É o limiar escolhido na seção 10 do `modelo_churn.ipynb` (menos falsos negativos e maior retorno líquido). Ele vem do próprio arquivo do modelo, não é digitado aqui. |
| **A base sai como veio, mais as colunas de previsão** | Quem usar o arquivo vê os dados originais do cliente ao lado do resultado, sem as transformações internas do modelo. |

---
### **1. Setup Inicial: Importando as Bibliotecas**
*Aqui não há treinamento, então bastam as bibliotecas de dados e o `joblib` para carregar o modelo. O
`scikit-learn` precisa ser a mesma versão usada para salvar o modelo (a do mesmo ambiente do
`modelo_churn.ipynb`).*

In [0]:
# --- Manipulação e Análise de Dados ---
import pandas as pd
import numpy as np
from pathlib import Path

# --- Carregamento do Modelo ---
import joblib

---
### **2. Carregando o Modelo**
*O `modelo_churn.ipynb` (seção 12.2) salva o modelo na pasta do usuário, porque a pasta do repositório não
aceita escrita. Já o app usa a cópia que está na pasta do projeto. A célula procura primeiro na pasta do
projeto (a mesma do app, para que as previsões batam) e, se não encontrar, na pasta do usuário.*

#### **2.1. Localizando o arquivo**

In [0]:
pasta_do_projeto = Path("/Workspace/Users/mateus@asn.rocks/Hackathon_Terra_Signal")

try:
    from databricks.sdk import WorkspaceClient
    pasta_do_usuario = Path(f"/Workspace/Users/{WorkspaceClient().current_user.me().user_name}")
except Exception:
    pasta_do_usuario = Path(".")

candidatos = [
    pasta_do_projeto / "modelo_churn.joblib",  # Opção 1: a mesma cópia usada pelo app
    pasta_do_usuario / "modelo_churn.joblib",  # Opção 2: a cópia salva pelo modelo_churn.ipynb
]

caminho_modelo = next((caminho for caminho in candidatos if caminho.exists()), None)
if caminho_modelo is None:
    raise FileNotFoundError("modelo_churn.joblib não encontrado. Rode antes a seção 12 do modelo_churn.ipynb.")

print(f"[INFO] Modelo encontrado em: {caminho_modelo}")

#### **2.2. Abrindo o artefato**
*O arquivo é um dicionário com três partes:*
* **`modelo`:** o `Pipeline` treinado (pré-processamento + Random Forest).
* **`limiar`:** a partir de qual probabilidade o cliente é apontado como risco.
* **`colunas`:** as colunas que o pipeline espera receber, na mesma ordem do treino.

In [0]:
artefato = joblib.load(caminho_modelo)

modelo_final = artefato["modelo"]
limiar_escolhido = artefato["limiar"]
colunas_do_modelo = artefato["colunas"]

print(f"[INFO] Limiar: {limiar_escolhido:.2f}")
print(f"[INFO] Colunas esperadas pelo modelo ({len(colunas_do_modelo)}): {colunas_do_modelo}")

---
### **3. Coleta de Dados**
*O `inference.csv` tem as mesmas colunas do `history.csv`, menos a `Churn`: são os clientes atuais, para os
quais ainda não se sabe se vão sair. A base original fica guardada em `df_original`, sem alterações, para
ser gravada no fim junto com a previsão.*

In [0]:
caminho_do_arquivo = pasta_do_projeto / "inference.csv"
df_original = pd.read_csv(caminho_do_arquivo)

print(f"{df_original.shape[0]} linhas x {df_original.shape[1]} colunas")

#### **3.1. Conferindo os mesmos problemas do histórico**
*No `history.csv`, `PhoneService` tinha `yes`/`no` em minúsculas e `tenure` tinha o texto `unknown`
(seções 3.3 e 3.4 do `modelo_churn.ipynb`). Conferimos se a carteira traz os mesmos problemas.*

In [0]:
print("PhoneService:", sorted(df_original["PhoneService"].astype(str).unique()))
print(f"tenure = 'unknown': {(df_original['tenure'].astype(str) == 'unknown').sum()} linhas")
print(f"TotalCharges vazio: {df_original['TotalCharges'].isna().sum()} linhas")

---
### **4. Preparação dos Dados**
*Os mesmos passos das seções 4 e 5 do `modelo_churn.ipynb`, reunidos numa função (a mesma
`preparar_dados` do `app.py`). Qualquer mudança na limpeza ou nas variáveis do modelo precisa ser repetida
aqui, senão o modelo recebe dados diferentes dos que viu no treino.*

| Passo | O que faz |
|---|---|
| **4.1** | `PhoneService`: `yes`/`no` → `Yes`/`No` |
| **4.2** | `tenure`: `unknown` → vazio |
| **4.3** | `TotalCharges` vazio com `tenure = 0` → `0` |
| **4.4** | `tenure` vazio → `TotalCharges ÷ MonthlyCharges` (0 a 72 meses) |
| **5.1** | `n_adicionais`: serviços de proteção e suporte contratados (0 a 4) |
| **5.2** | `inicio_relacionamento`: até 9 meses de casa |
| **5.3** | `mensal_fibra`: contrato mensal + fibra |
| **5.4** | `pagamento_automatico`: transferência ou cartão automáticos |
| **5.5** | `razao_cobranca`: `TotalCharges ÷ (tenure × MonthlyCharges)` |

In [0]:
def preparar_dados(df):
    df = df.copy()

    # --- 4.1. PhoneService: yes/no -> Yes/No ---
    df["PhoneService"] = df["PhoneService"].str.capitalize()

    # --- 4.2. tenure: "unknown" -> vazio ---
    df["tenure"] = pd.to_numeric(df["tenure"], errors="coerce")

    # --- 4.3. TotalCharges vazio com tenure = 0 -> 0 (cliente sem fatura fechada) ---
    sem_fatura = df["TotalCharges"].isna() & (df["tenure"] == 0)
    df.loc[sem_fatura, "TotalCharges"] = 0

    # --- 4.4. tenure vazio -> TotalCharges / MonthlyCharges (limitado a 0-72 meses) ---
    tenure_vazio = df["tenure"].isna()
    tenure_estimado = (df["TotalCharges"] / df["MonthlyCharges"]).round().clip(0, 72)
    df.loc[tenure_vazio, "tenure"] = tenure_estimado[tenure_vazio]

    # --- 5.1. n_adicionais: serviços de proteção e suporte contratados (0 a 4) ---
    df["n_adicionais"] = (
        (df["OnlineSecurity"] == "Yes").astype(int)
        + (df["OnlineBackup"] == "Yes").astype(int)
        + (df["DeviceProtection"] == "Yes").astype(int)
        + (df["TechSupport"] == "Yes").astype(int)
    )

    # --- 5.2. inicio_relacionamento: até 9 meses de casa ---
    df["inicio_relacionamento"] = np.where(df["tenure"] <= 9, "Yes", "No")
    df.loc[df["tenure"].isna(), "inicio_relacionamento"] = np.nan

    # --- 5.3. mensal_fibra: contrato mensal + fibra ---
    df["mensal_fibra"] = np.where(
        (df["Contract"] == "Month-to-month") & (df["InternetService"] == "Fiber optic"), "Yes", "No"
    )

    # --- 5.4. pagamento_automatico: transferência ou cartão automáticos ---
    df["pagamento_automatico"] = np.where(df["PaymentMethod"].str.contains("automatic"), "Yes", "No")

    # --- 5.5. razao_cobranca: TotalCharges / (tenure x MonthlyCharges) ---
    df["razao_cobranca"] = df["TotalCharges"] / (df["tenure"] * df["MonthlyCharges"])
    df["razao_cobranca"] = df["razao_cobranca"].replace([np.inf, -np.inf], np.nan)

    return df

#### **4.1. Aplicando a preparação**

In [0]:
df_preparado = preparar_dados(df_original)

print("PhoneService depois da limpeza:", sorted(df_preparado["PhoneService"].dropna().unique()))
print(f"tenure ainda vazio: {df_preparado['tenure'].isna().sum()} linhas")

#### **4.2. Conferindo as colunas e os valores ausentes**
*Antes de prever, garantimos que todas as colunas que o modelo espera existem na carteira. Os vazios que
restarem são preenchidos pelo `SimpleImputer` dentro do pipeline, com a mediana (numéricas) ou a moda
(categóricas) aprendidas no histórico, como na seção 4.5 do `modelo_churn.ipynb`.*

In [0]:
colunas_faltando = set(colunas_do_modelo) - set(df_preparado.columns)
if colunas_faltando:
    raise ValueError(f"Colunas que o modelo espera e não estão na carteira: {colunas_faltando}")

X_carteira = df_preparado[colunas_do_modelo]

print("[INFO] Todas as colunas esperadas pelo modelo estão presentes.")
print("\nValores ausentes que irão para o SimpleImputer do pipeline:")
print(X_carteira.isna().sum()[lambda s: s > 0])

---
### **5. Previsão**
*O pipeline devolve a probabilidade de cada cliente sair. Com o limiar do modelo, quem tem probabilidade
**≥ 0,20** é apontado como risco. Lembrando a escolha da seção 10 do `modelo_churn.ipynb`: o limiar baixo
aponta mais clientes (mais alarmes falsos) para deixar passar menos clientes que realmente saem.*

**Colunas novas:**

| Coluna | O que é |
|---|---|
| `probabilidade_churn` | Probabilidade de o cliente sair, de 0 a 1 |
| `churn_previsto` | `1` = risco de churn (probabilidade ≥ limiar), `0` = cliente estável |
| `churn_previsto_label` | O mesmo resultado em texto: `Sim` / `Não` (o padrão do app) |

In [0]:
probabilidade = modelo_final.predict_proba(X_carteira)[:, 1]

df_resultado = df_original.copy()
df_resultado["probabilidade_churn"] = probabilidade.round(4)
df_resultado["churn_previsto"] = (probabilidade >= limiar_escolhido).astype(int)
df_resultado["churn_previsto_label"] = np.where(df_resultado["churn_previsto"] == 1, "Sim", "Não")

print("[INFO] Previsão concluída.")

#### **5.1. Resultado da carteira**
*Quantos clientes o modelo aponta como risco. No conjunto de teste do `modelo_churn.ipynb`, com o mesmo
limiar, o modelo apontou cerca de metade dos clientes (547 de 1.127), então uma proporção parecida aqui é
esperada.*

In [0]:
n_clientes = len(df_resultado)
n_risco = df_resultado["churn_previsto"].sum()

print(f"Clientes na carteira: {n_clientes}")
print(f"Apontados como risco de churn: {n_risco} ({n_risco / n_clientes:.1%})".replace(".", ",", 1))
print(f"Apontados como estáveis: {n_clientes - n_risco} ({(n_clientes - n_risco) / n_clientes:.1%})".replace(".", ",", 1))

#### **5.2. Risco por tipo de contrato**
*Uma conferência de coerência: na exploração, o contrato mensal concentra a maior parte das saídas, então
ele deve aparecer com a maior proporção de clientes em risco.*

In [0]:
print("Proporção de clientes apontados como risco por contrato:")
print(df_resultado.groupby("Contract")["churn_previsto"].mean().round(3).sort_values(ascending=False))

---
### **6. Salvando a Base com a Previsão**
*O arquivo é gravado como `inference_com_previsao.csv`, primeiro na pasta do projeto (ao lado do
`inference.csv`). Como no `modelo_churn.ipynb` (seção 12.2), se a pasta do repositório não aceitar escrita,
o arquivo vai para a pasta do usuário.*

*Se o arquivo for gravado na pasta do projeto, ele aparece como alteração no Git da pasta.*

In [0]:
nome_do_arquivo = "inference_com_previsao.csv"

try:
    caminho_saida = pasta_do_projeto / nome_do_arquivo
    df_resultado.to_csv(caminho_saida, index=False)
except Exception as erro:
    print(f"[AVISO] Não foi possível gravar na pasta do projeto ({erro}). Gravando na pasta do usuário.")
    caminho_saida = pasta_do_usuario / nome_do_arquivo
    df_resultado.to_csv(caminho_saida, index=False)

print(f"[INFO] Base com a previsão salva em: {caminho_saida}")

#### **6.1. Conferindo o arquivo salvo**
*Lemos o arquivo de volta para garantir que ele tem o mesmo número de clientes da carteira original e as
colunas novas. Abaixo, os 10 clientes com maior probabilidade de sair: por eles começam as ligações de
retenção.*

In [0]:
df_conferencia = pd.read_csv(caminho_saida)

assert len(df_conferencia) == len(df_original), "O arquivo salvo não tem o mesmo número de clientes da carteira."
print(f"[INFO] Arquivo conferido: {df_conferencia.shape[0]} linhas x {df_conferencia.shape[1]} colunas")

colunas_resumo = ["customerID", "Contract", "tenure", "MonthlyCharges",
                  "probabilidade_churn", "churn_previsto_label"]
df_conferencia.sort_values("probabilidade_churn", ascending=False)[colunas_resumo].head(10)